In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Customer Churn Analysis - Week 1 EDA
**Author**: [Muhammad Umar Khan]
**Date**: [25/2/2026]
**Course**: Introduction to Applied AI
## Project Overview
This notebook performs exploratory data analysis on customer churn data to identify
patterns and inform our predictive modeling approach.
## Table of Contents
1. Dataset Overview
2. Numerical Features Analysis
3. Categorical Features Analysis
4. Feature Correlations
5. Key Insights and Findings


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Machine learning libraries
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# Settings
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
print('Libraries imported successfully!')

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/umarkhan360/dfbsdfb/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
# Check missing values
print('Missing values:')
print(df.isnull().sum())
# Convert TotalCharges to numeric (it may have spaces)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
# Fill missing values with median
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
# Verify no missing values remain
print(f'\nMissing values after cleaning: {df.isnull().sum().sum()}')

In [ ]:
# List of columns to encode
categorical_cols = ['gender', 'Partner', 'Dependents', 'PhoneService',
'MultipleLines', 'InternetService', 'OnlineSecurity',
'OnlineBackup', 'DeviceProtection', 'TechSupport',
'StreamingTV', 'StreamingMovies', 'Contract',
'PaperlessBilling', 'PaymentMethod']
# Create dummy variables
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print(f'Original columns: {df.shape[1]}')
print(f'After encoding: {df_encoded.shape[1]}')
print(f'New columns created: {df_encoded.shape[1] - df.shape[1]}')


In [ ]:
# Convert Churn to binary (Yes=1, No=0)
df_encoded['Churn'] = df_encoded['Churn'].map({'Yes': 1, 'No': 0})
# Verify encoding
print('Churn distribution:')
print(df_encoded['Churn'].value_counts())

In [ ]:
# Drop non-predictive columns
df_model = df_encoded.drop(['customerID'], axis=1)
# Separate features (X) and target (y)
X = df_model.drop('Churn', axis=1)
y = df_model['Churn']
print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'\nTarget distribution:')
print(y.value_counts(normalize=True))

In [ ]:
# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'\nChurn distribution in training set:')
print(y_train.value_counts(normalize=True))

In [ ]:
# Create and train the model
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)
# Make predictions
y_pred_lr = lr_model.predict(X_test)
# Calculate accuracy
lr_accuracy = accuracy_score(y_test, y_pred_lr)
print(f'Logistic Regression Accuracy: {lr_accuracy:.4f}')

In [ ]:
# Classification report
print('Classification Report:')
print(classification_report(y_test, y_pred_lr))
# Confusion matrix
cm_lr = confusion_matrix(y_test, y_pred_lr)
print('\nConfusion Matrix:')
print(cm_lr)

In [ ]:
# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Logistic Regression - Confusion Matrix')
plt.show()

In [ ]:
# Create and train decision tree
dt_model = DecisionTreeClassifier(random_state=42, max_depth=5)
dt_model.fit(X_train, y_train)
# Make predictions
y_pred_dt = dt_model.predict(X_test)
# Calculate accuracy
dt_accuracy = accuracy_score(y_test, y_pred_dt)
print(f'Decision Tree Accuracy: {dt_accuracy:.4f}')

In [ ]:
# Get feature importance
feature_importance = pd.DataFrame({
'feature': X_train.columns,
'importance': dt_model.feature_importances_
}).sort_values('importance', ascending=False)
# Display top 10 features
print('Top 10 Most Important Features:')
print(feature_importance.head(10))
# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'].head(10),
feature_importance['importance'].head(10))
plt.xlabel('Importance')
plt.title('Top 10 Feature Importances')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Create and train random forest
rf_model = RandomForestClassifier(
n_estimators=100, # Number of trees
random_state=42,
max_depth=10
)
rf_model.fit(X_train, y_train)
# Make predictions
y_pred_rf = rf_model.predict(X_test)
# Calculate accuracy
rf_accuracy = accuracy_score(y_test, y_pred_rf)
print(f'Random Forest Accuracy: {rf_accuracy:.4f}')

In [ ]:
# Classification report
print('Random Forest Classification Report:')
print(classification_report(y_test, y_pred_rf))
# Confusion matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()
plt.title('Random Forest - Confusion Matrix')

In [ ]:
# Create comparison dataframe
results = pd.DataFrame({
'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest'],
'Accuracy': [lr_accuracy, dt_accuracy, rf_accuracy]
})
# Sort by accuracy
results = results.sort_values('Accuracy', ascending=False)
print('Model Comparison:')
print(results)
# Find best model
best_model = results.iloc[0]['Model']
best_accuracy = results.iloc[0]['Accuracy']
print(f'\nBest Model: {best_model} with {best_accuracy:.4f} accuracy')

In [ ]:
# Plot model comparison
plt.figure(figsize=(10, 6))
plt.bar(results['Model'], results['Accuracy'], color=['#3498db', '#2ecc71', '#e74c3c'])

plt.xlabel('Model')
plt.ylabel('Accuracy')
plt.title('Model Performance Comparison')
plt.ylim(0.7, 0.85)  # Adjust based on your results
plt.xticks(rotation=45)

# Add value labels on bars
for i, v in enumerate(results['Accuracy']):
    plt.text(i, v + 0.005, f'{v:.4f}', ha='center')

plt.tight_layout()
plt.show()

In [ ]:
# Go back to the original dataframe before encoding
df_new = df.copy()
# Feature 1: Total Revenue (tenure * monthly charges)
df_new['TotalRevenue'] = df_new['tenure'] * df_new['MonthlyCharges']
# Feature 2: Count total services
service_cols = ['PhoneService', 'InternetService', 'OnlineSecurity',
'OnlineBackup', 'DeviceProtection', 'TechSupport',
'StreamingTV', 'StreamingMovies']
df_new['TotalServices'] = (df_new[service_cols] != 'No').sum(axis=1)
# Feature 3: Tenure groups
df_new['TenureGroup'] = pd.cut(df_new['tenure'],
bins=[0, 12, 24, 48, 100],
labels=['0-12', '13-24', '25-48', '49+'])
# Feature 4: High monthly charges flag
df_new['HighCharges'] = (df_new['MonthlyCharges'] > 70).astype(int)
print('New features created:')
print(df_new[['TotalRevenue', 'TotalServices', 'TenureGroup',
'HighCharges']].head())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

# --- 1. Separate target ---
if 'Churn' not in df_new.columns:
    raise ValueError("Column 'Churn' not found in df_new!")

y_new = df_new['Churn'].map({'Yes': 1, 'No': 0})

# --- 2. Encode only features ---
X_new = df_new.drop('Churn', axis=1)

# Encode categorical columns safely
categorical_cols = X_new.select_dtypes(include=['object', 'category']).columns
X_new = pd.get_dummies(X_new, columns=categorical_cols, drop_first=True)

# --- 3. Check if data exists ---
if X_new.shape[0] == 0 or y_new.shape[0] == 0:
    raise ValueError("No data to split! Check df_new after preprocessing.")

# --- 4. Split data ---
X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(
    X_new, y_new, test_size=0.2, random_state=42
)

# --- 5. Train model ---
rf_new = RandomForestClassifier(n_estimators=100, random_state=42)
rf_new.fit(X_train_new, y_train_new)

# --- 6. Predict & Accuracy ---
y_pred_new = rf_new.predict(X_test_new)
new_accuracy = accuracy_score(y_test_new, y_pred_new)

print(f'With New Features Accuracy: {new_accuracy:.4f}')

# Customer Churn Analysis Report

## What I Understood

## 1. Introduction

In this project, I analyzed a **Telecom Customer Churn dataset** to understand why customers leave a telecom company. The goal was to **predict whether a customer will churn (leave the service)** using machine learning models and to identify the most important factors affecting churn.

The dataset contains **7043 customers and 21 features**, including customer demographics, services used, billing information, and churn status.

---

## 2. Data Loading and Exploration

First, the dataset was loaded using **Pandas** and basic exploration was performed.

**Key observations:**

- Dataset size: **7043 rows and 21 columns**
- Target variable: **Churn**

**Churn values:**

- **No = Customer stayed**
- **Yes = Customer left**

This step helped in understanding the structure of the dataset.

---

## 3. Data Cleaning

Some preprocessing steps were required before building machine learning models.

### Missing Values

- The column **TotalCharges** had some non-numeric values (spaces).
- These values were converted to numeric.
- Missing values were filled using the **median value**.

After cleaning:

- **No missing values remained in the dataset.**

---

## 4. Data Preprocessing

Most columns in the dataset were **categorical** (e.g., gender, contract type, internet service).

To prepare the data for machine learning:

### One-Hot Encoding

Categorical variables were converted into numerical form using **dummy variables**.

**Results:**

- Original columns: **21**
- After encoding: **32 columns**

### Target Variable Conversion

The **Churn column** was converted into binary format:

| Value | Meaning |
|------|--------|
| 0 | No churn |
| 1 | Customer churn |

**Churn distribution:**

- **73.46% customers stayed**
- **26.54% customers left**

This shows the dataset is **slightly imbalanced**.

---

## 5. Feature Selection

The column **customerID** was removed because it does not help in prediction.

Then the dataset was split into:

- **Features (X)** → Customer information  
- **Target (y)** → Churn  

Total features used for training: **30**

---

## 6. Train-Test Split

The dataset was divided into:

- **80% Training data**
- **20% Testing data**

Training samples: **5634**  
Testing samples: **1409**

This helps evaluate how well the model performs on unseen data.

---

## 7. Machine Learning Models

Three different machine learning models were trained to predict churn.

### 7.1 Logistic Regression

Logistic Regression is a basic classification algorithm.

**Accuracy achieved:**

**80.34%**

Confusion matrix results showed:

- Many customers who stayed were correctly predicted.
- Some churn customers were missed.

---

### 7.2 Decision Tree

Decision Tree is a model that splits data based on important features.

**Accuracy achieved:**

**79.42%**

Feature importance analysis showed the most influential factors.

**Top important features:**

1. **Tenure**
2. **InternetService_Fiber optic**
3. **TotalCharges**
4. **PaymentMethod_Electronic check**
5. **MonthlyCharges**
6. **TechSupport**
7. **Contract type**

This means **customer duration and service type strongly influence churn**.

---

### 7.3 Random Forest

Random Forest is an ensemble model that combines many decision trees.

**Accuracy achieved:**

**80.70%**

This was the **best performing model** among the three.

---

## 8. Model Comparison

| Model | Accuracy |
|------|---------|
| Random Forest | **80.70%** |
| Logistic Regression | 80.34% |
| Decision Tree | 79.42% |

**Best model:**  
**Random Forest**

Because it gave the **highest prediction accuracy**.

---

## 9. Key Insights from the Analysis

### 1. Tenure is the most important factor
Customers with **shorter tenure are more likely to churn**.

### 2. Fiber optic users churn more
Customers with **fiber optic internet service** show higher churn rates.

### 3. Contract type affects churn
Customers with **month-to-month contracts churn more** compared to long-term contracts.

### 4. Payment method matters
Customers using **electronic check payments** have higher churn probability.

### 5. Monthly charges impact churn
Higher **monthly charges** increase the likelihood of churn.

---

## 10. Conclusion

From this analysis, I learned that **machine learning can effectively predict customer churn** using customer service and billing information.

Among the tested models, **Random Forest performed the best with about 80.7% accuracy**.

The most important factors influencing churn are:

- Customer tenure
- Internet service type
- Monthly charges
- Payment method
- Contract type

Understanding these factors can help telecom companies **identify at-risk customers and improve retention strategies**.